## 04 — Phase 2: Fine-Tune on Real Data

### 0. Setup

In [1]:
# Core imports
import shutil
import random
from pathlib import Path
from collections import Counter

import yaml
import torch

# -- Project root --------------------------------------------------------------
# Resolve to absolute. Ultralytics 8.4.39+ prepends a `runs/` dir to relative
# `project=` paths, which would save Phase 2 weights to the wrong folder.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = (NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR).resolve()
print(f"Project root: {PROJECT_ROOT}")

# -- Paths (all under data/) ---------------------------------------------------
DATA_DIR       = PROJECT_ROOT / "data"
FINETUNE_DIR   = DATA_DIR / "noodles_finetune_dataset"
TRAINING_DIR   = DATA_DIR / "noodles_training"

# -- Runtime device ------------------------------------------------------------
# Falls back to CPU on machines without CUDA (e.g. plain Windows without GPU).
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Training device: {DEVICE}")

# -- Shared constants (14 classes: 0-10 pieces, 11 board, 12 hinge, 13 pin) ---
PIECE_LABELS     = list("ABCDEFGHIJK")                              # piece classes 0-10
ALL_CLASS_NAMES  = PIECE_LABELS + ["board", "hinge", "pin"]         # classes 11, 12, 13
NUM_CLASSES      = len(ALL_CLASS_NAMES)                             # 14

PIECE_COLORS = {
    'A': ('Yellow',      (0xF9, 0xD6, 0x5E)),
    'B': ('SkyBlue',     (0x08, 0xA7, 0xE8)),
    'C': ('DarkBlue',    (0x20, 0x6D, 0xD9)),
    'D': ('Green',       (0x1F, 0xA1, 0x5B)),
    'E': ('Red',         (0xEE, 0x39, 0x4F)),
    'F': ('Teal',        (0x85, 0xDA, 0xBB)),
    'G': ('Pink',        (0xEC, 0x71, 0xA8)),
    'H': ('Purple',      (0xC7, 0x78, 0xB9)),
    'I': ('Orange',      (0xFC, 0x69, 0x0C)),
    'J': ('DarkRed',     (0xB6, 0x30, 0x48)),
    'K': ('YellowGreen', (0x95, 0xD4, 0x50)),
}

# Verify Phase 1 weights and files are available
p1_best = TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'best.pt'
finetune_yaml = FINETUNE_DIR / 'dataset.yaml'

print(f"\n  Phase 1 best: {p1_best} {'OK' if p1_best.exists() else 'missing'}")
print(f"  Fine-tune YAML: {finetune_yaml} {'OK' if finetune_yaml.exists() else 'missing'}")

Project root: C:\Users\abdul\Desktop\noodels
Training device: 0

  Phase 1 best: C:\Users\abdul\Desktop\noodels\data\noodles_training\phase1_synthetic\weights\best.pt OK
  Fine-tune YAML: C:\Users\abdul\Desktop\noodels\data\noodles_finetune_dataset\dataset.yaml OK


### 1. Inspect Available Datasets

In [2]:
for dataset_name in ['noodles_seg_dataset', 'noodles_finetune_dataset']:
    base = DATA_DIR / dataset_name
    print(f"\n📁 {dataset_name}/")
    for split in ['train', 'val']:
        img_dir = base / 'images' / split
        lbl_dir = base / 'labels' / split
        n_imgs = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
        n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f"  {split}: {n_imgs} images, {n_lbls} labels")
    
    yaml_file = base / 'dataset.yaml'
    if yaml_file.exists():
        with open(yaml_file) as f:
            cfg = yaml.safe_load(f)
        print(f"  YAML: nc={cfg.get('nc')}, names={list(cfg.get('names', {}).values())[:3]}...")
    else:
        print(f"  YAML: ❌ not found")


📁 noodles_seg_dataset/
  train: 0 images, 0 labels
  val: 0 images, 0 labels
  YAML: ❌ not found

📁 noodles_finetune_dataset/
  train: 101 images, 101 labels
  val: 0 images, 0 labels
  YAML: nc=14, names=['A_Yellow', 'B_SkyBlue', 'C_DarkBlue']...


### 2. Create Val Split from Train (if needed)

In [3]:
train_imgs = FINETUNE_DIR / 'images' / 'train'
val_imgs   = FINETUNE_DIR / 'images' / 'val'
train_lbls = FINETUNE_DIR / 'labels' / 'train'
val_lbls   = FINETUNE_DIR / 'labels' / 'val'

n_val = len(list(val_imgs.glob('*'))) if val_imgs.exists() else 0
n_train = len(list(train_imgs.glob('*'))) if train_imgs.exists() else 0

VAL_FRACTION = 0.2
SPLIT_SEED = 42  # deterministic split so re-runs don't reshuffle the val set

if n_val == 0 and n_train > 0:
    print(f"No val set found. Splitting {VAL_FRACTION*100:.0f}% from train ({n_train} images) with seed={SPLIT_SEED}...")

    val_imgs.mkdir(parents=True, exist_ok=True)
    val_lbls.mkdir(parents=True, exist_ok=True)

    all_imgs = sorted(train_imgs.glob('*'))
    n_val_target = max(1, int(len(all_imgs) * VAL_FRACTION))
    rng = random.Random(SPLIT_SEED)
    val_selection = rng.sample(all_imgs, n_val_target)

    for img_file in val_selection:
        shutil.move(str(img_file), str(val_imgs / img_file.name))
        lbl_src = train_lbls / (img_file.stem + '.txt')
        if lbl_src.exists():
            shutil.move(str(lbl_src), str(val_lbls / lbl_src.name))

    print(f"  Moved {len(val_selection)} images+labels to val")
    print(f"  Train: {len(list(train_imgs.glob('*')))} images remaining")
    print(f"  Val:   {len(list(val_imgs.glob('*')))} images")
else:
    print(f"Val set exists with {n_val} images. No split needed.")

No val set found. Splitting 20% from train (101 images) with seed=42...
  Moved 20 images+labels to val
  Train: 81 images remaining
  Val:   20 images


### 3. Fine-Tune (Phase 2)

In [5]:
from ultralytics import YOLO

# ── Load Phase 1 best weights ─────────────────────────────────────────────────
p1_best = TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'best.pt'
if not p1_best.exists():
    p1_best = TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'last.pt'
assert p1_best.exists(), f"Phase 1 weights not found! Run notebook 02 first."

model = YOLO(str(p1_best))
print(f"✅ Loaded Phase 1 weights: {p1_best}")

# Sanity check: Phase 1 head must already be 14-class (resized when nb 02 ran
# against the new dataset.yaml). If this fires, retrain Phase 1 first.
n_model = len(model.names)
assert n_model == NUM_CLASSES, (
    f"Phase 1 weights have {n_model} classes but dataset declares {NUM_CLASSES}. "
    f"Re-run notebook 02 to resize the head before fine-tuning."
)
print(f"   Phase 1 head: {n_model} classes ({list(model.names.values())})")

# ── Fine-tune ─────────────────────────────────────────────────────────────────
# Phase 2 fine-tunes on real photos. Synthetic Phase 1 supplies pieces (0-10),
# board (11), pin (13). Real Roboflow data refines the same classes on real
# photos. Hinge (12) is reserved in the schema but skipped at prep time
# (notebook 03, SKIP_HINGE=True) — flip that switch later to start training it.
results_phase2 = model.train(
    data=str(finetune_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2,
    patience=15,
    freeze=5,
    lr0=0.001,
    save=True,
    save_period=10,
    project=str(TRAINING_DIR),
    name='phase2_finetune',
    exist_ok=True,
    # Lighter augmentation for real data
    hsv_h=0.01,
    hsv_s=0.3,
    hsv_v=0.2,
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    mixup=0.05,
    copy_paste=0.1,
)

print("\n✅ Phase 2 fine-tuning complete!")

✅ Loaded Phase 1 weights: C:\Users\abdul\Desktop\noodels\data\noodles_training\phase1_synthetic\weights\best.pt
   Phase 1 head: 14 classes (['A_Yellow', 'B_SkyBlue', 'C_DarkBlue', 'D_Green', 'E_Red', 'F_Teal', 'G_Pink', 'H_Purple', 'I_Orange', 'J_DarkRed', 'K_YellowGreen', 'board', 'hinge', 'pin'])
New https://pypi.org/project/ultralytics/8.4.40 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.39  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\abdul\Desktop\noodels\data\noodles_finetune_dataset\dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, e

### 4. Validate & Compare

In [6]:
# Validate Phase 2 on finetune val set
metrics_p2 = model.val(
    data=str(finetune_yaml),
    imgsz=640,
    batch=16,
    device=DEVICE,
    plots=False,  # skip Ultralytics PR-curve plot (KeyErrors when a class has no preds)
)

print("\n=== Phase 2 Validation Metrics (Real Data) ===")
print(f"  Box  mAP@0.5:     {metrics_p2.box.map50:.4f}")
print(f"  Box  mAP@0.5:0.95: {metrics_p2.box.map:.4f}")
print(f"  Mask mAP@0.5:     {metrics_p2.seg.map50:.4f}")
print(f"  Mask mAP@0.5:0.95: {metrics_p2.seg.map:.4f}")

# Per-class breakdown — iterate ap_class_index, which lists the actual class
# ids that had val instances (not all 14 may be present in a small split).
print("\n=== Per-Class mAP@0.5 (Mask) ===")
ap50 = getattr(metrics_p2.seg, 'ap50', None)
ap_idx = getattr(metrics_p2.seg, 'ap_class_index', None)
if ap50 is not None and ap_idx is not None and len(ap50) > 0:
    for cls_id, ap in zip(ap_idx, ap50):
        name = model.names.get(int(cls_id), str(int(cls_id)))
        print(f"  {int(cls_id):2d} {name}: {ap:.4f}")
    missing = sorted(set(model.names.keys()) - set(int(c) for c in ap_idx))
    if missing:
        print(f"\n  (no val instances for: {[model.names[c] for c in missing]})")

# ── Compare with Phase 1 ──────────────────────────────────────────────────────
# Load Phase 1 results if available
p1_csv = TRAINING_DIR / 'phase1_synthetic' / 'results.csv'
if p1_csv.exists():
    import pandas as pd
    p1_results = pd.read_csv(p1_csv)
    last = p1_results.iloc[-1]
    map50_cols = [c for c in p1_results.columns if 'map50' in c.lower() and 'map50-95' not in c.lower()]
    if map50_cols:
        p1_map50 = last[map50_cols[0]]
        print(f"\n=== Comparison ===")
        print(f"  Phase 1 mAP50: {p1_map50:.4f} (synthetic val)")
        print(f"  Phase 2 mAP50: {metrics_p2.seg.map50:.4f} (real val)")
else:
    print(f"\n⚠️  Phase 1 results.csv not found at {p1_csv}")

Ultralytics 8.4.39  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
YOLO26n-seg summary (fused): 139 layers, 2,691,614 parameters, 0 gradients, 9.0 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 185.546.0 MB/s, size: 26.0 KB)
val: Scanning C:\Users\abdul\Desktop\noodels\data\noodles_finetune_dataset\labels\val.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.8it/s 0.7s1.6s
                   all         20        505      0.867      0.901       0.94      0.861      0.867      0.901       0.94      0.784
              A_Yellow          5          5      0.811      0.864      0.962      0.933      0.811      0.864      0.962      0.799
             B_SkyBlue          6          6      0.848          1      0.995      0.921      0.848          1      0.995      0.771
  